# Notebook 3 — Feature Engineering

Transforms the cleaned RASFF dataset into model-ready feature matrices for NB4 (Modeling).
All encoding decisions follow the feature selection table in **NB2 Section 7**.

## Pipeline Overview

| Section | Topic | Method | Cramér's V |
|---------|-------|--------|------------|
| 0 | Setup & shared utilities | — | — |
| 1 | Load data, define targets, chronological split | — | — |
| 2 | `hazards` — TF-IDF | TfidfVectorizer (top-200) | **0.376** |
| 3 | `hazard_substance` — Target encoding | TargetEncoder | **0.363** |
| 4 | `hazards_missing` — Pass-through binary | — | **0.275** |
| 5 | `subject` — Sentence embedding | all-MiniLM-L6-v2 | 0.298 |
| 6 | `year` — Raw integer | direct numeric | 0.243 |
| 7 | `hazard_tag` — OHE | get_dummies | 0.235 |
| 8 | `Hazard_Type`, `category`, `type` — OHE | get_dummies | 0.206 / 0.205 / 0.124 |
| 9 | `origin`, `notifying_country` — Target encoding | TargetEncoder | 0.190 / 0.170 |
| 10 | Assemble Scenario A & Scenario B feature matrices | concat | — |
| 11 | Save artifacts for NB4 | joblib | — |

## Two Scenarios (per NB1 / NB2 decisions)

| Scenario | `classification` included? | Use case |
|----------|--------------------------|----------|
| **A — Strict / Operational** | ❌ No | Predict at notification submission time (deployable) |
| **B — Relaxed / Upper-bound** | ✅ Yes | Performance ceiling, post-hoc analysis |

**Inputs:** `rasff_classified.csv` (with `Hazard_Type`, `hazard_tag`, `hazard_substance`, `hazards_missing`)  
**Outputs:** `X_train_A`, `X_test_A`, `X_train_B`, `X_test_B`, `y_train`, `y_test`, fitted encoders

> ⚠️ **Leakage prevention rules applied throughout:**  
> 1. Train/test split is **chronological** (by date), not random  
> 2. All encoders (`TfidfVectorizer`, `TargetEncoder`) are **fit on train only**  
> 3. Rare-category grouping (used in target encoding) is computed from **train data only** — never from the full dataset

---
## 0. Setup

In [ ]:
# ── Path Configuration ────────────────────────────────────────────
from pathlib import Path

# Resolves to rasff_risk_predictor/ root regardless of OS or user
ROOT_DIR  = Path.cwd().parent          # notebooks/ → rasff_risk_predictor/
DATA_DIR  = ROOT_DIR / "data"
MODEL_DIR = ROOT_DIR / "models"

# Verify folder structure on first run
for d in [DATA_DIR, MODEL_DIR]:
    if not d.exists():
        raise FileNotFoundError(
            f"Expected folder not found: {d}\n"
            "Please check your folder structure matches the README."
        )
# ──────────────────────────────────────────────────────────────────

%pip install sentence-transformers category_encoders --quiet

import pandas as pd
import numpy as np
import re
import warnings
import joblib
import json

from sklearn.feature_extraction.text import TfidfVectorizer
from category_encoders import TargetEncoder
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')

# ── Shared utilities ─────────────────────────────────────────────────────────
def clean_text(text):
    """Lowercase, strip slashes, separate digits from letters, collapse whitespace."""
    if text is None: return ''
    text = str(text).lower().strip()
    text = re.sub(r'/+', ' ', text)
    text = re.sub(r'(\d)([A-Za-z])', r'\1 \2', text)
    return re.sub(r'\s+', ' ', text).strip()

def recode_risk_2class(risk):
    """Binary target: serious (1) vs all others (0).
    Note: 'undecided' is mapped to 0 here. For 3-class modeling use recode_risk_3class."""
    if risk == 'serious': return 1
    if risk in ['no risk','not serious','potential risk','undecided','potentially serious']: return 0
    return -1

def recode_risk_3class(risk):
    """3-class target: Low / Medium / High. 'undecided' is dropped (caller filters out -1)."""
    if risk in ['no risk','not serious']: return 0           # Low
    if risk in ['potential risk','potentially serious']: return 1   # Medium
    if risk == 'serious': return 2                           # High
    return -1                                                # 'undecided' → exclude

def time_split(df, ratio=0.8):
    """Chronological split — prevents future data leaking into training."""
    df = df.copy().sort_values('date')
    cutoff = df.iloc[int(len(df) * ratio)]['date']
    return df[df['date'] <= cutoff].copy(), df[df['date'] > cutoff].copy()

def sanitize_cols(df):
    """Remove characters XGBoost/LightGBM reject in feature names."""
    df = df.copy()
    df.columns = (df.columns
        .str.replace(r'[\[\]{}:\"\',]', '_', regex=True)
        .str.replace(r'[^A-Za-z0-9_]+', '_', regex=True))
    return df

print('Utilities loaded.')


Note: you may need to restart the kernel to use updated packages.

Utilities loaded.


---
## 1. Load Data, Define Targets, Chronological Split

- Load `rasff_clean2.csv` (already contains `Hazard_Type`, `hazard_tag`, `hazard_substance`, `hazards_missing` from NB1)
- Build the binary target `risk_2class` (primary modeling target in NB4)
- **Chronological 80/20 split** by `date` — train on past, test on future

In [ ]:
DATA_PATH = DATA_DIR / "rasff_clean2.csv" 
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
print(f'Shape: {df.shape}')

# ── Reconstruct NB1 hazard features if not present (safety fallback) ────────
def extract_primary_tag(h):
    if pd.isnull(h): return 'unknown'
    tags = re.findall(r'\{([^}]+)\}', str(h))
    return tags[0].strip() if tags else 'unknown'

def extract_primary_substance(h):
    if pd.isnull(h): return 'unknown'
    parts = str(h).split(',')
    m = re.match(r'^(.+?)\s+-\s+\{', parts[0].strip())
    if m:
        s = m.group(1).strip().lower()
        s = re.sub(r'\s+(unauthorised substance|high level|migration|too high content'
                   r'|illegal trade.*|susp.*)', '', s).strip()
        return s
    return parts[0].strip().lower()[:80] if parts[0].strip() else 'unknown'

for col, fn in [('hazard_tag', extract_primary_tag),
                ('hazard_substance', extract_primary_substance)]:
    if col not in df.columns:
        df[col] = df['hazards'].apply(fn)
if 'hazards_missing' not in df.columns:
    df['hazards_missing'] = df['hazards'].isnull().astype(int)

# ── Cleaned text columns (for TF-IDF and embedding) ─────────────────────────
df['hazards_clean'] = df['hazards'].fillna('unknown').apply(clean_text)
df['subject_clean'] = df['subject'].fillna('').apply(clean_text)

# ── Define binary target ─────────────────────────────────────────────────────
df['risk_2class'] = df['risk_decision'].apply(recode_risk_2class)
df = df[df['risk_2class'] != -1].reset_index(drop=True)
print(f"Binary target: {df['risk_2class'].value_counts().to_dict()}")

# ── Chronological split ──────────────────────────────────────────────────────
train_df, test_df = time_split(df, ratio=0.8)
print(f'Train: {len(train_df):,}  ({train_df["date"].min().date()} → {train_df["date"].max().date()})')
print(f'Test:  {len(test_df):,}  ({test_df["date"].min().date()} → {test_df["date"].max().date()})')


Shape: (29984, 20)
Binary target: {1: 15839, 0: 14145}
Train: 23,988  (2020-01-21 → 2025-03-20)
Test:  5,996  (2025-03-20 → 2026-04-30)


---
## 2. `hazards` — TF-IDF (Cramér's V = 0.376)

**Encoding choice: TF-IDF** (not sentence embedding)

| Characteristic | Value | Implication |
|----------------|-------|-------------|
| Average length | 4.7 words | Too short for embedding to add semantic value |
| Vocabulary size | 951 tokens | Closed, domain-controlled RASFF term set |
| Text structure | `substance - {category}` | Structured, not natural language |
| Repetition | Top 5 strings account for 5,496 rows | High pattern frequency — TF-IDF excels |

`hazards` is a **structured, domain-controlled short-form field**, not free natural language.  
Its 951-token vocabulary consists of RASFF standardised substance names and category tags.  
TF-IDF captures these exact term patterns directly and efficiently.  
Sentence embedding adds no benefit here: `aflatoxin b1` and `aflatoxin total` are already  
linked by the shared token `aflatoxin` in TF-IDF space; no semantic inference is needed.  
*(Manning et al., Introduction to Information Retrieval, 2008: for domain-controlled vocabularies,  
TF-IDF matches or exceeds embedding-based methods.)*

~26% missing → filled with `'unknown'` token before encoding.

| TF-IDF parameter | Value | Rationale |
|-----------------|-------|----------|
| `max_features` | 200 | Covers dominant RASFF substances and categories |
| `ngram_range` | (1,2) | Bigrams preserve compound names: `aflatoxin b1`, `salmonella spp` |
| `sublinear_tf` | True | Dampens repeated mention skew |
| `min_df` | 5 | Removes rare/misspelled substance names |
| **Fit scope** | **Train only** | Prevents test vocabulary from influencing training |


In [3]:
TFIDF_MAX_FEATURES = 200

tfidf = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    sublinear_tf=True,
    ngram_range=(1, 2),
    min_df=5,
    token_pattern=r'[a-z][a-z]+'
)

tfidf.fit(train_df['hazards_clean'])  # fit on train only

train_haz_tfidf = pd.DataFrame(
    tfidf.transform(train_df['hazards_clean']).toarray(),
    columns=[f'haz_tfidf_{t}' for t in tfidf.get_feature_names_out()],
    index=train_df.index
)
test_haz_tfidf = pd.DataFrame(
    tfidf.transform(test_df['hazards_clean']).toarray(),
    columns=train_haz_tfidf.columns,
    index=test_df.index
)

print(f'TF-IDF shape: {train_haz_tfidf.shape}')
print(f'Top 10 terms: {tfidf.get_feature_names_out()[:10].tolist()}')


TF-IDF shape: (23988, 200)
Top 10 terms: ['acephate', 'acephate unauthorised', 'acetamiprid', 'acetamiprid pesticide', 'acid', 'additives', 'additives and', 'aflatoxin', 'aflatoxin mycotoxins', 'aflatoxin total']


---
## 3. `hazard_substance` — Target Encoding (Cramér's V = 0.363)

1,017 unique substances → OHE infeasible. Target encoding maps each substance to its
training-set serious-rate, with smoothing to prevent rare-substance overfit.

**Critical leakage prevention:**
- Rare-category grouping is computed from **train_df only** — never from full df
- `TargetEncoder.fit()` uses train only; `transform()` applied to both

In [4]:
# ── Group rare substances (< 20 cases in TRAIN) → 'other' ───────────────────
# CRITICAL: rarity must be computed from train only, not from train+test
train_sub_counts = train_df['hazard_substance'].value_counts()
rare_substances = train_sub_counts[train_sub_counts < 20].index
print(f'Rare substances (< 20 in train): {len(rare_substances):,} → grouped as "other"')

train_df['hazard_substance_grp'] = train_df['hazard_substance'].replace(rare_substances, 'other')
test_df['hazard_substance_grp']  = test_df['hazard_substance'].where(
    test_df['hazard_substance'].isin(train_sub_counts.index[train_sub_counts >= 20]), 'other'
)

# ── Target encode ───────────────────────────────────────────────────────────
te_substance = TargetEncoder(cols=['hazard_substance_grp'], smoothing=10)
te_substance.fit(train_df[['hazard_substance_grp']], train_df['risk_2class'])

train_sub_te = te_substance.transform(train_df[['hazard_substance_grp']])
train_sub_te.columns = ['hazard_substance_te']
train_sub_te.index = train_df.index

test_sub_te = te_substance.transform(test_df[['hazard_substance_grp']])
test_sub_te.columns = ['hazard_substance_te']
test_sub_te.index = test_df.index

print(f'Encoded train shape: {train_sub_te.shape}')
print(f'Sample encoded values: {train_sub_te["hazard_substance_te"].describe().to_dict()}')


Rare substances (< 20 in train): 798 → grouped as "other"
Encoded train shape: (23988, 1)
Sample encoded values: {'count': 23988.0, 'mean': 0.5488596026110651, 'std': 0.24739143655812448, 'min': 0.030353829396710455, '25%': 0.3423956500879578, '50%': 0.43373493975903615, '75%': 0.7857142857142121, 'max': 0.9824202432627289}


---
## 4. `hazards_missing` — Binary Pass-Through (Cramér's V = 0.275)

Already binary (0/1) from NB1 — no encoding needed.  
Cramér's V = 0.275, **stronger than `Hazard_Type` (0.206)** — independent predictive signal.

In [5]:
train_haz_miss = train_df[['hazards_missing']].copy()
test_haz_miss  = test_df[['hazards_missing']].copy()
print(f'hazards_missing — train: {train_haz_miss["hazards_missing"].mean():.1%} missing rate')
print(f'hazards_missing — test:  {test_haz_miss["hazards_missing"].mean():.1%} missing rate')


hazards_missing — train: 26.1% missing rate
hazards_missing — test:  26.3% missing rate


---
## 5. `subject` — Sentence Embedding (Cramér's V = 0.298)

**Encoding choice: Sentence Embedding** (not TF-IDF)

| Characteristic | Value | Implication |
|----------------|-------|-------------|
| Average length | 9.7 words | Long enough for sentence-level semantics |
| Vocabulary size | 6,928 tokens | 7.3× larger than `hazards` — high lexical diversity |
| Text structure | Free-form notification title | Natural language with synonyms and spelling variation |
| Synonym variation | 'from Turkiye' vs 'from Turkey', 'SESAME SEEDS' vs 'sesame' | TF-IDF treats these as different |

`subject` is a **free-text notification title** written by different notifying officers  
across 33 countries over 6 years. The same hazard can be described with different words:  
- `'SALMONELLA IN SESAME SEEDS FROM NIGERIA'` and `'Salmonella in sesame from Nigeria'`  
  are semantically identical but TF-IDF gives them a low similarity score.  
- `'Aflatoxins in dried figs from Turkiye'` and `'Aflatoxins in dried figs from Turkey'`  
  (62 and 61 occurrences respectively) — same event, different country name spelling.  

Sentence embedding (`all-MiniLM-L6-v2`) maps both variations to nearby vectors,  
preserving semantic proximity that TF-IDF would miss.  
*(Reimers & Gurevych, EMNLP 2019: SBERT sentence embeddings outperform TF-IDF  
on semantic similarity tasks, especially with high lexical diversity.)*

Encoding is computed independently on train and test — the SBERT model has no  
trainable weights updated here, so there is no leakage risk.


In [6]:
EMBED_MODEL = 'all-MiniLM-L6-v2'
embedder = SentenceTransformer(EMBED_MODEL)

def embed_series(series, prefix, idx):
    vecs = embedder.encode(series.tolist(), show_progress_bar=True, normalize_embeddings=False)
    return pd.DataFrame(vecs,
                        columns=[f'{prefix}_{i}' for i in range(vecs.shape[1])],
                        index=idx)

train_sub_emb = embed_series(train_df['subject_clean'], 'sub_emb', train_df.index)
test_sub_emb  = embed_series(test_df['subject_clean'],  'sub_emb', test_df.index)

print(f'Subject embedding: {train_sub_emb.shape[1]}-dim')


Batches:   0%|          | 0/750 [00:00<?, ?it/s]

Batches:   0%|          | 0/188 [00:00<?, ?it/s]

Subject embedding: 384-dim


---
## 6. `year` — Raw Integer (Cramér's V = 0.243)

Per NB2: serious rate declines 67.3% (2020) → 44.6% (2026).  
Tree-based models (XGBoost / LightGBM / CatBoost) handle ordinal integers natively
without scaling. **No normalization applied** — preserves interpretability.

In [7]:
train_year = train_df[['year']].copy().astype(int)
test_year  = test_df[['year']].copy().astype(int)
print(f'year range — train: {train_year["year"].min()} → {train_year["year"].max()}')
print(f'year range — test:  {test_year["year"].min()} → {test_year["year"].max()}')


year range — train: 2020 → 2025
year range — test:  2025 → 2026


---
## 7. `hazard_tag` — One-Hot Encoding (Cramér's V = 0.235)

30 RASFF internal tag categories. Mid-level granularity between Hazard_Type (9)
and hazard_substance (1,017). Test reindexed to match train vocabulary.

In [8]:
tag_ohe_train = pd.get_dummies(train_df['hazard_tag'], prefix='tag').astype(int)
tag_ohe_test  = pd.get_dummies(test_df['hazard_tag'],  prefix='tag').astype(int)
tag_ohe_test  = tag_ohe_test.reindex(columns=tag_ohe_train.columns, fill_value=0)
tag_ohe_train.index = train_df.index
tag_ohe_test.index  = test_df.index
print(f'hazard_tag OHE: {tag_ohe_train.shape[1]} cols')


hazard_tag OHE: 30 cols


---
## 8. `Hazard_Type`, `category`, `type`, `classification` — One-Hot Encoding

All four are low-cardinality categoricals. `category` switched from 384-dim sentence
embedding to OHE per NB2 — 37 structured labels do not benefit from semantic embedding.

| Column | Cramér's V | Cardinality | Scenario |
|--------|-----------|-------------|----------|
| `Hazard_Type` | 0.206 | 9 | A + B |
| `category` | 0.205 | 37 | A + B |
| `type` | 0.124 | 6 | A + B |
| `classification` | 0.307 | 5 | **B only** (post-decision leakage) |

In [9]:
def ohe_train_test(col, prefix, train_df, test_df, fill_na='missing'):
    """Fit OHE on train, reindex test to match."""
    train_ohe = pd.get_dummies(train_df[col].fillna(fill_na), prefix=prefix).astype(int)
    test_ohe  = pd.get_dummies(test_df[col].fillna(fill_na),  prefix=prefix).astype(int)
    test_ohe  = test_ohe.reindex(columns=train_ohe.columns, fill_value=0)
    train_ohe.index = train_df.index
    test_ohe.index  = test_df.index
    return train_ohe, test_ohe

train_haz_ohe,  test_haz_ohe  = ohe_train_test('Hazard_Type',    'hazard',   train_df, test_df)
train_cat_ohe,  test_cat_ohe  = ohe_train_test('category',       'cat',      train_df, test_df)
train_type_ohe, test_type_ohe = ohe_train_test('type',           'type',     train_df, test_df)
train_clf_ohe,  test_clf_ohe  = ohe_train_test('classification', 'clf',      train_df, test_df)  # Scenario B only

print(f'Hazard_Type OHE    : {train_haz_ohe.shape[1]} cols')
print(f'category OHE       : {train_cat_ohe.shape[1]} cols')
print(f'type OHE           : {train_type_ohe.shape[1]} cols')
print(f'classification OHE : {train_clf_ohe.shape[1]} cols  (Scenario B only)')


Hazard_Type OHE    : 9 cols
category OHE       : 37 cols
type OHE           : 6 cols
classification OHE : 5 cols  (Scenario B only)


---
## 9. `origin`, `notifying_country` — Target Encoding

| Column | Cramér's V | Unique values | Why TE? |
|--------|-----------|---------------|---------|
| `notifying_country` | 0.190 | 33 | Skewed — top 3 (Germany, NL, Italy) = 35% of cases |
| `origin` | 0.170 | 640 | OHE would create 640 sparse columns |

Same leakage rules as Section 3: rare-category grouping uses **train counts only**.

In [10]:
def rare_group_train_only(col, train_df, test_df, threshold=50):
    """Group rare categories to 'other' using TRAIN counts only.
    Returns modified copies of train_df and test_df with new column `{col}_grp`."""
    train_df = train_df.copy()
    test_df  = test_df.copy()
    train_df[col] = train_df[col].fillna('missing')
    test_df[col]  = test_df[col].fillna('missing')
    counts = train_df[col].value_counts()
    keep   = counts[counts >= threshold].index
    train_df[f'{col}_grp'] = train_df[col].where(train_df[col].isin(keep), 'other')
    test_df[f'{col}_grp']  = test_df[col].where(test_df[col].isin(keep),  'other')
    n_rare = (counts < threshold).sum()
    print(f'  {col}: {n_rare} rare categories (< {threshold} in train) grouped to "other"')
    return train_df, test_df

print('Rare-category grouping (train counts only):')
train_df, test_df = rare_group_train_only('origin',            train_df, test_df, threshold=50)
train_df, test_df = rare_group_train_only('notifying_country', train_df, test_df, threshold=50)

te_geo_cols = ['origin_grp', 'notifying_country_grp']
te_geo = TargetEncoder(cols=te_geo_cols, smoothing=10)
te_geo.fit(train_df[te_geo_cols], train_df['risk_2class'])

train_geo_te = te_geo.transform(train_df[te_geo_cols])
train_geo_te.columns = ['origin_te', 'notifying_country_te']
train_geo_te.index = train_df.index

test_geo_te = te_geo.transform(test_df[te_geo_cols])
test_geo_te.columns = ['origin_te', 'notifying_country_te']
test_geo_te.index = test_df.index

print(f'\nTarget-encoded geo features: {train_geo_te.shape}')
print(train_geo_te.describe().round(3).to_string())


Rare-category grouping (train counts only):
  origin: 492 rare categories (< 50 in train) grouped to "other"
  notifying_country: 2 rare categories (< 50 in train) grouped to "other"

Target-encoded geo features: (23988, 2)
       origin_te  notifying_country_te
count  23988.000             23988.000
mean       0.548                 0.548
std        0.119                 0.136
min        0.137                 0.170
25%        0.485                 0.453
50%        0.577                 0.559
75%        0.620                 0.632
max        0.871                 0.757


---
## 10. Assemble Scenario A & Scenario B Feature Matrices

| Block | Feature | Cramér's V | Encoding | Cols | A | B |
|-------|---------|-----------|----------|------|---|---|
| 1 | `hazards` | 0.376 | TF-IDF top-200 | 200 | ✅ | ✅ |
| 2 | `hazard_substance` | 0.363 | Target encoding | 1 | ✅ | ✅ |
| 3 | `hazards_missing` | 0.275 | Binary | 1 | ✅ | ✅ |
| 4 | `subject` | 0.298 | Sentence embedding | 384 | ✅ | ✅ |
| 5 | `year` | 0.243 | Raw integer | 1 | ✅ | ✅ |
| 6 | `hazard_tag` | 0.235 | OHE | ~30 | ✅ | ✅ |
| 7 | `Hazard_Type` | 0.206 | OHE | 9 | ✅ | ✅ |
| 8 | `category` | 0.205 | OHE | ~37 | ✅ | ✅ |
| 9 | `notifying_country` | 0.190 | TE | 1 | ✅ | ✅ |
| 10 | `origin` | 0.170 | TE | 1 | ✅ | ✅ |
| 11 | `type` | 0.124 | OHE | 6 | ✅ | ✅ |
| 12 | `classification` | 0.307 | OHE | 5 | ❌ | ✅ |

In [11]:
# ── Scenario A — operational (no classification) ────────────────────────────
common_train_blocks = [
    train_haz_tfidf,   # (1) hazards TF-IDF      200
    train_sub_te,      # (2) hazard_substance TE   1
    train_haz_miss,    # (3) hazards_missing       1
    train_sub_emb,     # (4) subject embedding   384
    train_year,        # (5) year                  1
    tag_ohe_train,     # (6) hazard_tag OHE       ~30
    train_haz_ohe,     # (7) Hazard_Type OHE       9
    train_cat_ohe,     # (8) category OHE        ~37
    train_geo_te,      # (9-10) origin + notif. TE 2
    train_type_ohe,    # (11) type OHE             6
]
common_test_blocks = [
    test_haz_tfidf, test_sub_te, test_haz_miss, test_sub_emb,
    test_year, tag_ohe_test, test_haz_ohe, test_cat_ohe,
    test_geo_te, test_type_ohe,
]

X_train_A = sanitize_cols(pd.concat(common_train_blocks, axis=1))
X_test_A  = sanitize_cols(pd.concat(common_test_blocks,  axis=1))

# ── Scenario B — relaxed (adds classification) ──────────────────────────────
X_train_B = sanitize_cols(pd.concat(common_train_blocks + [train_clf_ohe], axis=1))
X_test_B  = sanitize_cols(pd.concat(common_test_blocks  + [test_clf_ohe],  axis=1))

y_train = train_df['risk_2class']
y_test  = test_df['risk_2class']

print(f'Scenario A — operational  : X_train {X_train_A.shape}, X_test {X_test_A.shape}')
print(f'Scenario B — upper-bound  : X_train {X_train_B.shape}, X_test {X_test_B.shape}')
print(f'\ny_train: {y_train.value_counts().to_dict()}')
print(f'y_test : {y_test.value_counts().to_dict()}')
print(f'\nTrain class balance: {y_train.mean():.1%} serious')
print(f'Test  class balance: {y_test.mean():.1%} serious')


Scenario A — operational  : X_train (23988, 671), X_test (5996, 671)
Scenario B — upper-bound  : X_train (23988, 676), X_test (5996, 676)

y_train: {1: 13157, 0: 10831}
y_test : {0: 3314, 1: 2682}

Train class balance: 54.8% serious
Test  class balance: 44.7% serious


---
## 11. Save Artifacts for NB4

Persists the two scenario feature matrices, fitted encoders, and metadata.

In [ ]:
OUT_DIR = MODEL_DIR  
import os
os.makedirs(OUT_DIR, exist_ok=True)

# Feature matrices for both scenarios
joblib.dump((X_train_A, X_test_A, y_train, y_test), OUT_DIR / "nb3_splits_A.pkl")
joblib.dump((X_train_B, X_test_B, y_train, y_test), OUT_DIR / "nb3_splits_B.pkl")

# Fitted encoders (needed at inference time)
joblib.dump(tfidf,        OUT_DIR / "tfidf_vectorizer.pkl")
joblib.dump(te_substance, OUT_DIR / "te_substance.pkl")
joblib.dump(te_geo,       OUT_DIR / "te_geo.pkl")

# OHE column references (to reindex new data at inference)
ohe_columns = {
    'hazard_tag':     tag_ohe_train.columns.tolist(),
    'Hazard_Type':    train_haz_ohe.columns.tolist(),
    'category':       train_cat_ohe.columns.tolist(),
    'type':           train_type_ohe.columns.tolist(),
    'classification': train_clf_ohe.columns.tolist(),
}
joblib.dump(ohe_columns, OUT_DIR / "ohe_columns.pkl")

# Metadata describing what's in each scenario
preprocess_info = {
    'sentence_model'       : EMBED_MODEL,
    'tfidf_max_features'   : TFIDF_MAX_FEATURES,
    'split_strategy'       : 'chronological 80/20',
    'rare_threshold_substance': 20,
    'rare_threshold_geo'   : 50,
    'scenario_A_features'  : list(X_train_A.columns),
    'scenario_B_features'  : list(X_train_B.columns),
    'scenario_A_n_features': X_train_A.shape[1],
    'scenario_B_n_features': X_train_B.shape[1],
    'feature_priority': [
        {'feature': 'hazards',           'cramers_v': 0.376, 'encoding': 'TF-IDF top-200'},
        {'feature': 'hazard_substance',  'cramers_v': 0.363, 'encoding': 'TargetEncoding'},
        {'feature': 'classification',    'cramers_v': 0.307, 'encoding': 'OHE',
         'note': 'Scenario B only (post-decision leakage)'},
        {'feature': 'subject',           'cramers_v': 0.298, 'encoding': 'SentenceEmbedding-384'},
        {'feature': 'hazards_missing',   'cramers_v': 0.275, 'encoding': 'binary pass-through'},
        {'feature': 'year',              'cramers_v': 0.243, 'encoding': 'raw integer'},
        {'feature': 'hazard_tag',        'cramers_v': 0.235, 'encoding': 'OHE'},
        {'feature': 'Hazard_Type',       'cramers_v': 0.206, 'encoding': 'OHE'},
        {'feature': 'category',          'cramers_v': 0.205, 'encoding': 'OHE'},
        {'feature': 'notifying_country', 'cramers_v': 0.190, 'encoding': 'TargetEncoding'},
        {'feature': 'origin',            'cramers_v': 0.170, 'encoding': 'TargetEncoding'},
        {'feature': 'type',              'cramers_v': 0.124, 'encoding': 'OHE'},
    ],
    'excluded_features': [
        {'feature': 'month',        'cramers_v': 0.030, 'reason': 'near-zero association'},
        {'feature': 'date',         'reason': 'superseded by year'},
        {'feature': 'distribution', 'reason': 'post-decision; 32% missing'},
        {'feature': 'forAttention', 'reason': 'post-decision; 44% missing'},
        {'feature': 'forFollowUp',  'reason': 'post-decision; 50% missing'},
        {'feature': 'reference',    'reason': 'record ID'},
        {'feature': 'operator',     'reason': '5,207 unique values; no signal'},
    ]
}
with open(OUT_DIR / "preprocess_info.json", 'w') as f:
    json.dump(preprocess_info, f, indent=2)

print('Saved artifacts:')
print(f'  nb3_splits_A.pkl       — Scenario A: {X_train_A.shape[1]} features')
print(f'  nb3_splits_B.pkl       — Scenario B: {X_train_B.shape[1]} features')
print(f'  tfidf_vectorizer.pkl, te_substance.pkl, te_geo.pkl, ohe_columns.pkl')
print(f'  preprocess_info.json   — feature manifest')


Saved artifacts:
  nb3_splits_A.pkl       — Scenario A: 671 features
  nb3_splits_B.pkl       — Scenario B: 676 features
  tfidf_vectorizer.pkl, te_substance.pkl, te_geo.pkl, ohe_columns.pkl
  preprocess_info.json   — feature manifest
